In [20]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import csv
from datetime import datetime
import re
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from tqdm import tqdm

def clear_gpu_memory():
    """Clear GPU memory"""
    torch.cuda.empty_cache()
    gc.collect()

def load_final_model():
    """Load the final model with both adapters optimized for GPU"""
    model_name = "Qwen/Qwen2.5-1.5B-Instruct"
    
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    print("Loading base model with GPU optimization...")
    clear_gpu_memory()
    
    # Load base model with optimized settings for GPU
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        offload_state_dict=False,
    )
    
    # Force model to GPU
    print("Moving model to GPU...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    base_model.to(device)
    
    print("Loading both adapters...")
    model = PeftModel.from_pretrained(
        base_model,
        "./qwen1.5b-clinical-reasoning/final_adapters",
        device_map="auto",
        torch_dtype=torch.float16,
        offload_state_dict=False,
    )
    
    # Ensure model is in evaluation mode
    model.eval()
    
    # Force model to stay on GPU
    if torch.cuda.is_available():
        model = model.to('cuda')
        print(f"Model loaded on device: {model.device}")
        print(f"GPU Memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    
    return model, tokenizer, device

def generate_medical_note(row):
    """Generate a clinical note from a dataframe row in a single line format"""
    note_parts = []
    
    # Patient info
    age = int(row['Age']) if pd.notna(row['Age']) else 'unknown'
    gender = str(row['Gender']).lower() if pd.notna(row['Gender']) else 'unknown'
    note_parts.append(f"{age} year old {gender}")
    
    # Symptoms
    symptoms = []
    for symptom_col in ['Symptom_1', 'Symptom_2', 'Symptom_3']:
        if pd.notna(row[symptom_col]) and str(row[symptom_col]).strip():
            symptoms.append(str(row[symptom_col]).lower())
    
    if symptoms:
        if len(symptoms) == 1:
            symptoms_text = symptoms[0]
        else:
            symptoms_text = ", ".join(symptoms[:-1]) + " and " + symptoms[-1]
        note_parts.append(f"presents with {symptoms_text}")
    
    # Vital signs
    vitals = []
    
    # Heart rate
    if pd.notna(row['Heart_Rate_bpm']):
        try:
            hr = int(float(row['Heart_Rate_bpm']))
            vitals.append(f"heart rate {hr} bpm")
        except:
            pass
    
    # Temperature
    if pd.notna(row['Body_Temperature_C']):
        try:
            temp = float(row['Body_Temperature_C'])
            vitals.append(f"temperature {temp}°C")
        except:
            pass
    
    # Blood pressure
    if pd.notna(row['Blood_Pressure_mmHg']):
        bp = str(row['Blood_Pressure_mmHg'])
        vitals.append(f"blood pressure {bp} mmHg")
    
    # Oxygen saturation
    if pd.notna(row['Oxygen_Saturation_%']):
        try:
            oxy = int(float(row['Oxygen_Saturation_%']))
            vitals.append(f"oxygen saturation {oxy}%")
        except:
            pass
    
    if vitals:
        if len(vitals) == 1:
            vitals_text = vitals[0]
        else:
            vitals_text = ", ".join(vitals[:-1]) + " and " + vitals[-1]
        note_parts.append(f"with vital signs: {vitals_text}")
    
    # Severity (if available)
    if 'Severity' in row and pd.notna(row['Severity']):
        severity = str(row['Severity']).lower()
        note_parts.append(f"condition is {severity}")
    
    # Join all parts into a single line
    clinical_note = ". ".join(note_parts) + "."
    
    return clinical_note

def extract_disease_from_response(text):
    """Simple but effective disease extraction from LLM response"""
    
    # Clean the text
    text = text.strip()
    
    # Common disease patterns to look for FIRST
    disease_patterns = [
        # Pattern 1: **Most Suspected Disease:** Disease Name
        (r'\*\*Most Suspected Disease:\*\*\s*([^\n]+)', 0),
        (r'Most Suspected Disease:\s*([^\n]+)', 0),
        
        # Pattern 2: **Most Suspected Disease:** Disease Name (with parentheses)
        (r'\*\*Most Suspected Disease:\*\*\s*([^(]+?)\s*(?:\([^)]+\))?', 0),
        
        # Pattern 3: ### Most Suspected Diseases: (for bullet lists)
        (r'### Most Suspected Diseases:\s*\n\s*[*-]\s*\*\*([^*\n]+)\*\*', 0),
        (r'### Most Suspected Diseases:\s*\n\s*[*-]\s*([^\n]+)', 0),
        
        # Pattern 4: **Most Suspicious Disease:** (common typo)
        (r'\*\*Most Suspicious Disease:\*\*\s*([^\n]+)', 0),
        (r'Most Suspicious Disease:\s*([^\n]+)', 0),
        
        # Pattern 5: First bolded text that looks like a disease
        (r'\*\*([A-Z][a-z]+(?:\s+[A-Za-z]+){0,3})\*\*', 1),
        
        # Pattern 6: First line after cleaning
        (r'^([A-Z][a-z]+(?:\s+[A-Za-z]+){0,4})', 2),
        
        # Pattern 7: Disease mentioned in first sentence
        (r'^[^.:;!?]*?\b([A-Z][a-z]+(?:\s+[A-Za-z]+){0,3})\b', 3),
    ]
    
    # Try each pattern in order
    for pattern, priority in disease_patterns:
        matches = re.findall(pattern, text, re.MULTILINE | re.IGNORECASE)
        if matches:
            # Take the first match
            disease = matches[0].strip()
            
            # Clean up common prefixes/suffixes
            disease = re.sub(r'^(?:Acute|Chronic|Severe|Mild|Moderate)\s+', '', disease, flags=re.IGNORECASE)
            disease = re.sub(r'\s*\([^)]+\)$', '', disease)  # Remove parentheses
            disease = re.sub(r'\s*\[[^\]]+\]$', '', disease)  # Remove brackets
            
            # Skip if it's just formatting or instructions
            skip_words = ['name', 'disease', 'diagnosis', 'suspected', 'most', 'list', 'other', 'possible', 'significant', 'risks']
            if any(word in disease.lower() for word in skip_words) or len(disease.split()) > 6:
                continue
                
            if disease and len(disease) > 2:
                return disease
    
    # If no pattern matched, try a simpler approach
    lines = text.split('\n')
    for line in lines[:5]:  # Check first 5 lines
        line = line.strip()
        if line and not line.startswith(('#', '*', '-', '1.', '2.', '3.', '4.', '**')):
            # Look for capital letter words (likely disease names)
            words = line.split()
            if len(words) >= 2:
                # Check if first word is capitalized (likely a disease name)
                if words[0][0].isupper():
                    # Take first 2-4 words
                    potential_disease = ' '.join(words[:min(4, len(words))])
                    # Clean up
                    potential_disease = re.sub(r'[:;.,]$', '', potential_disease)
                    if len(potential_disease) > 3:
                        return potential_disease
    
    # Last resort: return first non-empty line that doesn't look like formatting
    for line in lines:
        line = line.strip()
        if line and len(line) > 5 and not any(char in line for char in '#*-'):
            return line[:50]  # Return first 50 chars
    
    return "Unknown"

def extract_other_sections(text):
    """Extract other sections from the response"""
    sections = {
        'other_risks': "None",
        'diagnostic_reasoning': "None",
        'precautions': "None"
    }
    
    # Extract other risks
    other_risk_patterns = [
        r'\*\*Other Significant Risks:\*\*\s*(.+?)(?=\n\s*\*\*|\n\s*$|\Z)',
        r'Other Significant Risks:\s*(.+?)(?=\n\s*(?:Diagnostic|Precautions|\Z))',
        r'\*\*Other Possible Diseases:\*\*\s*(.+?)(?=\n\s*\*\*|\n\s*$|\Z)',
        r'Other Possible Diseases:\s*(.+?)(?=\n\s*(?:Diagnostic|Precautions|\Z))',
    ]
    
    for pattern in other_risk_patterns:
        match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        if match:
            sections['other_risks'] = match.group(1).strip()[:200]  # Limit length
            break
    
    # Extract diagnostic reasoning
    reasoning_patterns = [
        r'\*\*Diagnostic Reasoning:\*\*\s*(.+?)(?=\n\s*\*\*|\n\s*$|\Z)',
        r'Diagnostic Reasoning:\s*(.+?)(?=\n\s*Precautions|\Z)',
        r'\*\*Diagnostic Reasons:\*\*\s*(.+?)(?=\n\s*\*\*|\n\s*$|\Z)',
        r'Diagnostic Reasons:\s*(.+?)(?=\n\s*Precautions|\Z)',
    ]
    
    for pattern in reasoning_patterns:
        match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        if match:
            sections['diagnostic_reasoning'] = match.group(1).strip()[:300]  # Limit length
            break
    
    # Extract precautions
    precaution_patterns = [
        r'\*\*Precautions:\*\*\s*(.+?)(?=\n\s*$|\Z)',
        r'Precautions:\s*(.+?)(?=\n\s*$|\Z)',
        r'\*\*Preparations:\*\*\s*(.+?)(?=\n\s*$|\Z)',
        r'Preparations:\s*(.+?)(?=\n\s*$|\Z)',
    ]
    
    for pattern in precaution_patterns:
        match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        if match:
            sections['precautions'] = match.group(1).strip()[:200]  # Limit length
            break
    
    return sections

def run_inference_on_sample(model, tokenizer, clinical_note, device):
    """Run inference on a single clinical note with GPU optimization"""
    # Clear instruction with exact format
    instruction = """You are a medical diagnostic assistant. Analyze the clinical note and provide:
1. Most suspected disease
2. Other diseases with significant risk  
3. Diagnostic reasoning
4. Precautions for the most suspected disease

Format your response EXACTLY as follows:
**Most Suspected Disease:** [name of the disease]
**Other Significant Risks:** [list other possible diseases]
**Diagnostic Reasoning:** [explain your reasoning step by step]
**Precautions:** [recommend precautions for the suspected disease]

Be concise and clinical in your response."""
    
    prompt = f"<|im_start|>user\n{instruction}\n\nClinical Note:\n{clinical_note}<|im_end|>\n<|im_start|>assistant\n"
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)
    
    # Move inputs to GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Generate with optimized settings
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.3,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.2,
                no_repeat_ngram_size=3,
                early_stopping=True,
            )
    
    # Decode response WITHOUT skipping special tokens
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Extract ONLY the assistant's response
    assistant_start = "<|im_start|>assistant"
    if assistant_start in full_response:
        # Get everything after the last occurrence of assistant_start
        parts = full_response.rsplit(assistant_start, 1)
        if len(parts) > 1:
            assistant_part = parts[1]
            # Remove any trailing end tokens
            if "<|im_end|>" in assistant_part:
                assistant_part = assistant_part.split("<|im_end|>")[0]
            assistant_part = assistant_part.strip()
        else:
            assistant_part = full_response.strip()
    else:
        # If no assistant tag found, try to find after the last user message
        user_end = "<|im_end|>\n<|im_start|>assistant"
        if user_end in full_response:
            assistant_part = full_response.split(user_end)[-1].strip()
            if "<|im_end|>" in assistant_part:
                assistant_part = assistant_part.split("<|im_end|>")[0].strip()
        else:
            assistant_part = full_response.strip()
    
    # Remove any remaining special tokens
    assistant_part = assistant_part.replace("<|im_end|>", "").replace("<|im_start|>", "").strip()
    
    # Clear memory
    del inputs, outputs
    clear_gpu_memory()
    
    return assistant_part

class ResultSaver:
    """Class to save results in real-time to multiple files"""
    
    def __init__(self, timestamp):
        self.timestamp = timestamp
        self.full_results_file = f"full_results_{timestamp}.txt"
        self.summary_results_file = f"summary_results_{timestamp}.csv"
        
        # Initialize files
        self._init_files()
        
    def _init_files(self):
        """Initialize output files with headers"""
        # Text file for full results (human-readable)
        if not os.path.exists(self.full_results_file):
            with open(self.full_results_file, 'w', encoding='utf-8') as f:
                f.write("MEDICAL DIAGNOSIS MODEL EVALUATION RESULTS\n")
                f.write("=" * 80 + "\n\n")
                f.write("Note: Skipping patients with 'Healthy' diagnosis\n\n")
        
        # CSV file for summary results
        if not os.path.exists(self.summary_results_file):
            with open(self.summary_results_file, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow([
                    'patient_id', 'age', 'gender', 'actual_diagnosis',
                    'predicted_disease', 'match', 'clinical_note_summary'
                ])
    
    def save_result(self, result, skip_healthy=True):
        """Save a single result to both files"""
        # Skip if diagnosis is Healthy and skip_healthy is True
        if skip_healthy and result['actual_diagnosis'].lower() == 'healthy':
            return
        
        # Save to TEXT file (full details, human-readable)
        with open(self.full_results_file, 'a', encoding='utf-8') as f:
            f.write(f"\n{'='*80}\n")
            f.write(f"PATIENT ID: {result['patient_id']}\n")
            f.write(f"AGE/GENDER: {result['age']} years, {result['gender']}\n")
            f.write(f"CLINICAL NOTE:\n{result['clinical_note']}\n\n")
            f.write(f"ACTUAL DIAGNOSIS: {result['actual_diagnosis']}\n")
            f.write(f"PREDICTED DISEASE: {result['extracted_suspected_disease']}\n")
            f.write(f"MATCH: {'✓' if result.get('match', False) else '✗'}\n\n")
            f.write("LLM RESPONSE:\n")
            f.write("-" * 40 + "\n")
            f.write(f"{result['llm_full_response']}\n\n")
            f.write("EXTRACTED SECTIONS:\n")
            f.write("-" * 40 + "\n")
            f.write(f"Most Suspected Disease: {result['extracted_suspected_disease']}\n")
            f.write(f"Other Risks: {result['other_risks'][:100]}...\n" if len(result['other_risks']) > 100 else f"Other Risks: {result['other_risks']}\n")
            f.write(f"Diagnostic Reasoning: {result['diagnostic_reasoning'][:150]}...\n" if len(result['diagnostic_reasoning']) > 150 else f"Diagnostic Reasoning: {result['diagnostic_reasoning']}\n")
            f.write(f"Precautions: {result['precautions'][:100]}...\n" if len(result['precautions']) > 100 else f"Precautions: {result['precautions']}\n")
        
        # Save to CSV (summary)
        with open(self.summary_results_file, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            
            # Shorten clinical note for CSV
            clinical_note_summary = result['clinical_note'][:150] + "..." if len(result['clinical_note']) > 150 else result['clinical_note']
            
            writer.writerow([
                result['patient_id'],
                result['age'],
                result['gender'],
                result['actual_diagnosis'],
                result['extracted_suspected_disease'],
                result.get('match', False),
                clinical_note_summary
            ])

def check_match(predicted, actual):
    """Check if predicted disease matches actual diagnosis"""
    if not predicted or predicted.lower() in ['unknown', 'none', '']:
        return False
    
    predicted_lower = str(predicted).lower().strip()
    actual_lower = str(actual).lower().strip()
    
    # Direct match
    if actual_lower in predicted_lower or predicted_lower in actual_lower:
        return True
    
    # Disease mappings for common variations
    disease_mappings = {
        'flu': ['influenza', 'flu', 'viral influenza', 'influenza virus'],
        'cold': ['common cold', 'cold', 'upper respiratory infection', 'uri', 'viral cold'],
        'pneumonia': ['pneumonia', 'community-acquired pneumonia', 'bacterial pneumonia'],
        'bronchitis': ['bronchitis', 'acute bronchitis', 'chronic bronchitis'],
        'healthy': ['healthy', 'no disease', 'normal', 'no diagnosis', 'well']
    }
    
    # Check through mappings
    for base_disease, variations in disease_mappings.items():
        if actual_lower == base_disease:
            # Check if predicted contains any variation
            if any(var in predicted_lower for var in variations):
                return True
        elif predicted_lower == base_disease:
            # Check if actual contains any variation
            if any(var in actual_lower for var in variations):
                return True
    
    # Check word overlap
    predicted_words = set(predicted_lower.split())
    actual_words = set(actual_lower.split())
    common_words = predicted_words.intersection(actual_words)
    
    if len(common_words) >= 1 and len(actual_words) <= 3:
        return True
    
    return False

def normalize_disease_name(name):
    """Normalize disease names for comparison"""
    if not name or str(name).lower() in ['unknown', 'none', '']:
        return 'Unknown'
    
    name_lower = str(name).lower().strip()
    
    # Normalization rules
    if any(word in name_lower for word in ['flu', 'influenza']):
        return 'Flu'
    elif any(word in name_lower for word in ['cold', 'uri', 'upper respiratory']):
        return 'Cold'
    elif 'pneumonia' in name_lower:
        return 'Pneumonia'
    elif 'bronchitis' in name_lower:
        return 'Bronchitis'
    elif any(word in name_lower for word in ['healthy', 'normal', 'no disease', 'well', 'none']):
        return 'Healthy'
    elif any(word in name_lower for word in ['myocardial', 'heart', 'ami', 'cardiac']):
        return 'Cardiac'
    elif any(word in name_lower for word in ['embolism', 'pe', 'pulmonary']):
        return 'Pulmonary_Embolism'
    elif 'mononucleosis' in name_lower or 'mono' in name_lower or 'ebv' in name_lower:
        return 'Mononucleosis'
    elif 'streptococcal' in name_lower or 'strep' in name_lower:
        return 'Strep_Throat'
    elif 'respiratory' in name_lower and 'infection' in name_lower:
        return 'URI'
    else:
        # Take first 2-3 meaningful words
        words = [w for w in name_lower.split() if len(w) > 2]
        if words:
            return '_'.join(words[:2]).title()
        else:
            return 'Other'

def evaluate_model():
    """Main evaluation function with GPU optimization and real-time saving"""
    
    # Check GPU availability
    if not torch.cuda.is_available():
        print("WARNING: CUDA not available. Running on CPU may be slow!")
    
    # Load model and tokenizer
    print("Loading model with GPU optimization...")
    model, tokenizer, device = load_final_model()
    
    # Load dataset from CSV
    print("Loading dataset...")
    try:
        df = pd.read_csv('disease_diagnosis.csv')
        print(f"Successfully loaded {len(df)} samples from disease_dataset.csv")
        
        # Filter out Healthy diagnoses for evaluation
        df_non_healthy = df[df['Diagnosis'].str.lower() != 'healthy']
        print(f"Filtering out 'Healthy' diagnoses: {len(df_non_healthy)}/{len(df)} samples remaining")
        
    except FileNotFoundError:
        print("ERROR: disease_dataset.csv not found!")
        print("Please ensure the CSV file is in the current directory.")
        return [], None
    
    # Initialize result saver
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    result_saver = ResultSaver(timestamp)
    
    # Initialize metrics tracking
    results = []
    predicted_diseases = []
    actual_diseases = []
    
    print(f"\nEvaluating on {len(df_non_healthy)} non-healthy samples...")
    print("=" * 80)
    
    # Process samples with progress bar
    for idx, row in tqdm(df_non_healthy.iterrows(), total=len(df_non_healthy), desc="Processing samples"):
        # Generate medical note in single line format
        clinical_note = generate_medical_note(row)
        
        # Run inference
        try:
            llm_response = run_inference_on_sample(model, tokenizer, clinical_note, device)
            
            # Extract disease with improved method
            predicted_disease = extract_disease_from_response(llm_response)
            
            # Extract other sections
            other_sections = extract_other_sections(llm_response)
            
            # Get actual diagnosis
            actual_diagnosis = str(row['Diagnosis']) if 'Diagnosis' in row and pd.notna(row['Diagnosis']) else 'Unknown'
            
            # Check if prediction matches actual diagnosis
            match = check_match(predicted_disease, actual_diagnosis)
            
            # Create result entry
            result_entry = {
                'patient_id': int(row.get('Patient_ID', idx + 1)),
                'age': int(row.get('Age', 0)) if pd.notna(row.get('Age')) else 0,
                'gender': str(row.get('Gender', 'Unknown')),
                'clinical_note': clinical_note,
                'actual_diagnosis': actual_diagnosis,
                'llm_full_response': llm_response,
                'extracted_suspected_disease': predicted_disease,
                'other_risks': other_sections['other_risks'],
                'diagnostic_reasoning': other_sections['diagnostic_reasoning'],
                'precautions': other_sections['precautions'],
                'match': match
            }
            
            # Save result immediately (skip healthy ones)
            result_saver.save_result(result_entry, skip_healthy=True)
            results.append(result_entry)
            
            # Store for metrics
            predicted_diseases.append(predicted_disease)
            actual_diseases.append(actual_diagnosis)
            
            # Print progress
            if idx < 10:  # Show first 10 for debugging
                print(f"Sample {idx+1}: Actual='{actual_diagnosis}', Predicted='{predicted_disease}', Match={'✓' if match else '✗'}")
            
            # Clear memory periodically
            if (idx + 1) % 10 == 0:
                clear_gpu_memory()
                
        except Exception as e:
            print(f"\nError processing sample {idx + 1}: {str(e)}")
            # Save error result
            error_result = {
                'patient_id': int(row.get('Patient_ID', idx + 1)),
                'age': int(row.get('Age', 0)) if pd.notna(row.get('Age')) else 0,
                'gender': str(row.get('Gender', 'Unknown')),
                'clinical_note': clinical_note,
                'actual_diagnosis': str(row.get('Diagnosis', 'Unknown')),
                'llm_full_response': f"ERROR: {str(e)}",
                'extracted_suspected_disease': "Error",
                'other_risks': "None",
                'diagnostic_reasoning': "None",
                'precautions': "None",
                'match': False
            }
            result_saver.save_result(error_result, skip_healthy=True)
            continue
    
    # Calculate and save metrics
    print("\n" + "="*80)
    print("CALCULATING METRICS...")
    print("="*80)
    
    if len(results) > 0:
        calculate_and_save_metrics(results, predicted_diseases, actual_diseases, timestamp)
    else:
        print("No results to evaluate!")
    
    # Clean up
    clear_gpu_memory()
    
    return results, timestamp

def calculate_and_save_metrics(results, predicted_diseases, actual_diseases, timestamp):
    """Calculate and save evaluation metrics"""
    
    # Normalize diseases for comparison
    normalized_predicted = [normalize_disease_name(d) for d in predicted_diseases]
    normalized_actual = [normalize_disease_name(d) for d in actual_diseases]
    
    # Get unique classes
    all_classes = list(set(normalized_actual + normalized_predicted))
    unique_classes = [c for c in all_classes if c != 'Unknown']
    
    # Calculate simple accuracy based on matches in results
    matches = [r['match'] for r in results if 'match' in r]
    accuracy = sum(matches) / len(matches) if matches else 0
    
    # Calculate additional metrics if we have multiple classes
    if len(unique_classes) > 1:
        try:
            # Filter out Unknown for metric calculation
            filtered_actual = []
            filtered_predicted = []
            for a, p in zip(normalized_actual, normalized_predicted):
                if a != 'Unknown' and p != 'Unknown':
                    filtered_actual.append(a)
                    filtered_predicted.append(p)
            
            if len(set(filtered_actual)) > 1 and len(filtered_actual) > 0:
                precision = precision_score(filtered_actual, filtered_predicted, 
                                           average='macro', zero_division=0)
                recall = recall_score(filtered_actual, filtered_predicted, 
                                     average='macro', zero_division=0)
                f1 = f1_score(filtered_actual, filtered_predicted, 
                              average='macro', zero_division=0)
                
                # Generate classification report
                class_report = classification_report(filtered_actual, filtered_predicted, 
                                                     zero_division=0)
                
                # Confusion matrix
                cm = confusion_matrix(filtered_actual, filtered_predicted, 
                                     labels=sorted(set(filtered_actual + filtered_predicted)))
                cm_labels = sorted(set(filtered_actual + filtered_predicted))
            else:
                precision = recall = f1 = accuracy
                class_report = "Insufficient data for detailed metrics"
                cm = None
                cm_labels = []
                
        except Exception as e:
            print(f"Error calculating advanced metrics: {e}")
            precision = recall = f1 = accuracy
            class_report = f"Could not generate detailed report: {e}"
            cm = None
            cm_labels = []
    else:
        precision = recall = f1 = accuracy
        class_report = "Single class in dataset or insufficient data"
        cm = None
        cm_labels = []
    
    # Save metrics report
    report_file = f"metrics_report_{timestamp}.txt"
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write("MEDICAL DIAGNOSIS MODEL EVALUATION METRICS\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Evaluation Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total Samples Evaluated: {len(results)} (Healthy diagnoses excluded)\n")
        f.write(f"Device Used: {'GPU' if torch.cuda.is_available() else 'CPU'}\n\n")
        
        f.write("OVERALL METRICS:\n")
        f.write("-" * 30 + "\n")
        f.write(f"Accuracy: {accuracy:.4f} ({accuracy:.2%})\n")
        f.write(f"Precision (Macro): {precision:.4f}\n")
        f.write(f"Recall (Macro): {recall:.4f}\n")
        f.write(f"F1-Score (Macro): {f1:.4f}\n\n")
        
        f.write("DETAILED CLASSIFICATION REPORT:\n")
        f.write("-" * 30 + "\n")
        f.write(class_report + "\n")
        
        if cm is not None and len(cm_labels) > 0:
            f.write("\nCONFUSION MATRIX:\n")
            f.write("-" * 30 + "\n")
            f.write("Rows: Actual, Columns: Predicted\n\n")
            # Create header
            f.write(f"{'Actual/Predicted':<20}" + "".join([f"{label:<15}" for label in cm_labels]) + "\n")
            f.write("-" * (20 + 15 * len(cm_labels)) + "\n")
            # Write matrix
            for i, label in enumerate(cm_labels):
                f.write(f"{label:<20}")
                for j in range(len(cm_labels)):
                    f.write(f"{cm[i][j] if i < len(cm) and j < len(cm[i]) else 0:<15}")
                f.write("\n")
    
    # Create visualization if we have a confusion matrix
    if cm is not None and len(cm_labels) > 1:
        try:
            plt.figure(figsize=(10, 8))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                        xticklabels=cm_labels, yticklabels=cm_labels)
            plt.title(f'Confusion Matrix - Accuracy: {accuracy:.2%}')
            plt.ylabel('Actual Diagnosis')
            plt.xlabel('Predicted Diagnosis')
            plt.tight_layout()
            plt.savefig(f'confusion_matrix_{timestamp}.png', dpi=300, bbox_inches='tight')
            plt.close()
            print(f"Confusion matrix saved to: confusion_matrix_{timestamp}.png")
        except Exception as e:
            print(f"Could not create visualization: {e}")
    
    print(f"\nMETRICS SUMMARY:")
    print(f"Samples processed: {len(results)} (Healthy diagnoses excluded)")
    print(f"Accuracy: {accuracy:.2%}")
    if len(unique_classes) > 1:
        print(f"Precision: {precision:.2%}")
        print(f"Recall: {recall:.2%}")
        print(f"F1-Score: {f1:.2%}")
    print(f"\nDetailed metrics saved to: {report_file}")
    
    # Show some example predictions
    print("\nSAMPLE PREDICTIONS:")
    print("-" * 40)
    for i, result in enumerate(results[:min(5, len(results))]):
        match_symbol = "✓" if result['match'] else "✗"
        print(f"{i+1}. Patient {result['patient_id']}: Actual='{result['actual_diagnosis']}', Predicted='{result['extracted_suspected_disease'][:30]}...' {match_symbol}")

def analyze_results(timestamp):
    """Analyze saved results"""
    summary_file = f"summary_results_{timestamp}.csv"
    full_file = f"full_results_{timestamp}.txt"
    
    print("\n" + "="*80)
    print("ANALYZING RESULTS")
    print("="*80)
    
    if os.path.exists(summary_file):
        try:
            df_summary = pd.read_csv(summary_file)
            print(f"\nSummary Statistics:")
            print(f"Total samples (non-healthy): {len(df_summary)}")
            
            if 'match' in df_summary.columns:
                correct = df_summary['match'].sum()
                total = len(df_summary)
                accuracy = correct / total if total > 0 else 0
                print(f"Correct predictions: {correct}/{total} ({accuracy:.2%})")
            
            # Analysis by actual diagnosis
            if 'actual_diagnosis' in df_summary.columns:
                print("\nPerformance by Actual Diagnosis:")
                for diagnosis in sorted(df_summary['actual_diagnosis'].unique()):
                    subset = df_summary[df_summary['actual_diagnosis'] == diagnosis]
                    if len(subset) > 0:
                        correct_subset = subset['match'].sum() if 'match' in subset.columns else 0
                        accuracy_subset = correct_subset / len(subset) if len(subset) > 0 else 0
                        print(f"  {diagnosis:<15}: {len(subset):>3} samples, Accuracy: {accuracy_subset:.2%}")
            
            # Show some examples
            print("\nSample predictions:")
            sample_size = min(5, len(df_summary))
            for i in range(sample_size):
                row = df_summary.iloc[i]
                match_symbol = "✓" if row.get('match', False) else "✗"
                predicted = str(row['predicted_disease'])[:30]
                print(f"  {i+1}. Patient {row['patient_id']}: Actual='{row['actual_diagnosis']}', Predicted='{predicted}...' {match_symbol}")
                
        except Exception as e:
            print(f"Error reading summary file: {e}")
    
    if os.path.exists(full_file):
        print(f"\nFull results available in: {full_file}")
        print(f"Summary results available in: {summary_file}")

if __name__ == "__main__":
    print("="*80)
    print("MEDICAL DIAGNOSIS MODEL EVALUATION SYSTEM")
    print("="*80)
    
    # Check GPU info
    if torch.cuda.is_available():
        print(f"GPU Available: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    else:
        print("WARNING: Running on CPU - this will be slow!")
    
    # Run evaluation
    results, timestamp = evaluate_model()
    
    # Analyze results
    if timestamp:
        analyze_results(timestamp)
    
    print("\n" + "="*80)
    print("EVALUATION COMPLETE!")
    if timestamp:
        print(f"Timestamp: {timestamp}")
        print(f"Files created:")
        print(f"  - full_results_{timestamp}.txt (complete LLM responses in readable format)")
        print(f"  - summary_results_{timestamp}.csv (summary information)")
        print(f"  - metrics_report_{timestamp}.txt (evaluation metrics)")
        if os.path.exists(f'confusion_matrix_{timestamp}.png'):
            print(f"  - confusion_matrix_{timestamp}.png (visualization)")
    print("="*80)

MEDICAL DIAGNOSIS MODEL EVALUATION SYSTEM
GPU Available: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU Memory: 5.67 GB
Loading model with GPU optimization...
Loading tokenizer...
Loading base model with GPU optimization...
Moving model to GPU...
Loading both adapters...
Model loaded on device: cuda:0
GPU Memory allocated: 5.16 GB
Loading dataset...
Successfully loaded 2000 samples from disease_dataset.csv
Filtering out 'Healthy' diagnoses: 833/2000 samples remaining

Evaluating on 833 non-healthy samples...


Processing samples:   0%|          | 0/833 [00:00<?, ?it/s]/tmp/ipykernel_2189/4027328730.py:300: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Processing samples:   0%|          | 1/833 [00:08<1:54:24,  8.25s/it]

Sample 1: Actual='Flu', Predicted='Infectious Mononucleosis', Match=✗


Processing samples:   0%|          | 2/833 [00:13<1:29:01,  6.43s/it]

Sample 6: Actual='Flu', Predicted='Streptococcal Pharyngitis', Match=✗


...

Sample 7: Actual='Bronchitis', Predicted='Streptococcal pharyngitis', Match=✗


Processing samples:   0%|          | 4/833 [00:30<1:47:56,  7.81s/it]

Sample 9: Actual='Cold', Predicted='Respiratory Distress Syndrome', Match=✗
Sample 10: Actual='Flu', Predicted='Pneumonia', Match=✗


Processing samples: 100%|██████████| 833/833 [2:33:41<00:00, 11.07s/it]  



CALCULATING METRICS...
Confusion matrix saved to: confusion_matrix_20251214_143641.png

METRICS SUMMARY:
Samples processed: 833 (Healthy diagnoses excluded)
Accuracy: 9.96%
Precision: 3.32%
Recall: 1.10%
F1-Score: 1.45%

Detailed metrics saved to: metrics_report_20251214_143641.txt

SAMPLE PREDICTIONS:
----------------------------------------
1. Patient 1: Actual='Flu', Predicted='Infectious Mononucleosis...' ✗
2. Patient 6: Actual='Flu', Predicted='Streptococcal Pharyngitis...' ✗
3. Patient 7: Actual='Bronchitis', Predicted='Streptococcal pharyngitis...' ✗
4. Patient 9: Actual='Cold', Predicted='Respiratory Distress Syndrome...' ✗
5. Patient 10: Actual='Flu', Predicted='Pneumonia...' ✗

ANALYZING RESULTS

Summary Statistics:
Total samples (non-healthy): 833
Correct predictions: 83/833 (9.96%)

Performance by Actual Diagnosis:
  Bronchitis     : 334 samples, Accuracy: 5.69%
  Cold           : 163 samples, Accuracy: 20.86%
  Flu            : 292 samples, Accuracy: 8.90%
  Pneumonia    

In [21]:
clear_gpu_memory()